In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_path = "NEU-DET/train/images"
val_path   = "NEU-DET/validation/images"

preprocess = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(train_path, transform=preprocess)
val_dataset   = datasets.ImageFolder(val_path, transform=preprocess)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)


def train_and_evaluate(model, model_type):
    print(f"\nRunning model: {model_type}")

    for param in model.parameters():
        param.requires_grad = False

    if model_type == "resnet":
        model.fc = nn.Linear(model.fc.in_features, 6)

    elif model_type == "efficientnet":
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, 6)

    elif model_type == "inception":
        model.fc = nn.Linear(model.fc.in_features, 6)

    model = model.to(device)

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-3
    )
    criterion = nn.CrossEntropyLoss()

    epochs = 10
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        sample_count = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)

            if model_type == "inception" and isinstance(outputs, tuple):
                main_out, aux_out = outputs
                loss = criterion(main_out, labels) + 0.4 * criterion(aux_out, labels)
            else:
                loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * inputs.size(0)
            sample_count += inputs.size(0)

        print(f"Epoch {epoch + 1} | Loss: {epoch_loss / sample_count:.4f}")

    model.eval()
    correct_preds = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)

            if isinstance(outputs, tuple):
                outputs = outputs[0]

            predictions = torch.argmax(outputs, dim=1)
            correct_preds += (predictions.cpu() == labels).sum().item()
            total_samples += labels.size(0)

    accuracy = correct_preds / total_samples
    print(f"Validation Accuracy: {accuracy:.4f}")

    return accuracy


resnet_model = models.resnet50(pretrained=True)
efficientnet_model = models.efficientnet_b0(pretrained=True)
inception_model = models.inception_v3(pretrained=True)

performance = {}

performance["ResNet50"] = train_and_evaluate(resnet_model, "resnet")
performance["EfficientNetB0"] = train_and_evaluate(efficientnet_model, "efficientnet")
performance["InceptionV3"] = train_and_evaluate(inception_model, "inception")

print("\n--- Model Performance Summary ---")
for model_name, acc in performance.items():
    print(f"{model_name}: {acc:.4f}")

best_model = max(performance, key=performance.get)
print(f"\nTop Performing Model: {best_model}")